## Setup

In [69]:
from arcgis.gis import GIS
import arcpy
from arcgis.features import FeatureLayer
import pandas as pd
import json
import urllib.request
import urllib.parse

In [111]:
gis = GIS('pro')
print(gis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
arcpy.env.addOutputsToMap = False

GIS @ https://www.arcgis.com/ version:[2026, 2]


## Content Access

### GIS Content Searches and Misc Imports

In [14]:
# Just doing some searching for Tijuana River data
results1 = gis.content.search("Tijuana River Water Quality",item_type = 'Feature Layer', outside_org=True, max_items=20)

for item in results1:
    print(item.title, '|', item.owner, '|', item.type)

SDH_streams | DerekSu_IvyMax | Feature Service
Tijuana River Valley Water Quality_WFL1 | nstatom_UCSDOnline | Feature Service
TRI_Environmental | jack.daley_4walls | Feature Service
TRI_Environmental | jack.daley_4walls | Feature Service
NOAA NERR Estuary Map_WFL1 | dordoming0235@ucla.edu_gisucla | Feature Service


In [33]:
# Let's look at that second result
service1 = results1[1]
for layer in service1.layers:
    print(layer.properties.name)

TJRV_Monitoring_Stations
Tijuana River Valley Reserve Boundary
Tijuana River Valley Reserve Habitats
Tijuana River Valley Watershed NERRS
Tijuana River Valley Reserve Habitats Accuracy Assessment
San Diego County Annual Precipitation Contours SANGIS
San Diego County Watershed SANGIS


In [63]:
# Maybe I'll add this, maybe not

# monitoring = service1.layers[0]
# map.addDataFromPath(monitoring.url)
# map.removeLayer(map.listLayers()[0])

In [85]:
# Okay I'm adding a watershed layer in case I want to use that later,
# as well as a CA counties layer in case I need to spatial join

service2 = FeatureLayer("https://geo.sandag.org/server/rest/services/"
                        "Hosted/Watersheds/FeatureServer")
# map.addDataFromPath(service2.url)
service3 = FeatureLayer("https://services7.arcgis.com/yiuFazTjHE8F5gzQ/arcgis/"
                        "rest/services/California_Counties/FeatureServer")
# map.addDataFromPath(service3.url)

### Beach Monitoring Imports

In [84]:
# Add monitoring stations layer
url = "https://data.ca.gov/api/3/action/datastore_search?resource_id=98e628ff-d012-4982-ad32-b9f9ad8ab524&limit=2000"
fileobj = urllib.request.urlopen(url)
response_dict = json.loads(fileobj.read())
mon_stat = pd.DataFrame(response_dict['result']['records'])
mon_stat_sedf = mon_stat.spatial.from_xy(mon_stat, x_column='Station_UpperLon', y_column = 'Station_UpperLat')

gdb_path = aprx.defaultGeodatabase
mon_stat_sedf.spatial.to_featureclass(location=fr"{gdb_path}\monitoring_stations")

map.addDataFromPath(fr"{gdb_path}\monitoring_stations")

In [113]:
# Add sampe_data layer filtered by date and county name
resource_id = "7bd961cf-abe4-433b-8033-378161237ff3"

sql = f'''
SELECT * FROM "{resource_id}"
WHERE "County" = 'San Diego'
AND "SampleDate"::date >= '2025-01-01'
'''

url = f"https://data.ca.gov/api/3/action/datastore_search_sql?sql={urllib.parse.quote(sql)}"
fileobj = urllib.request.urlopen(url)
response_dict = json.loads(fileobj.read())

sample_data = pd.DataFrame(response_dict['result']['records'])
sample_data_sedf = sample_data.spatial.from_xy(sample_data, x_column='Station_UpperLon', y_column = 'Station_UpperLat')

gdb_path = aprx.defaultGeodatabase
sample_data_sedf.spatial.to_featureclass(location=fr"{gdb_path}\sample_data")

map.addDataFromPath(fr"{gdb_path}\sample_data")

In [119]:
# Add beach_closures layer filtered by date and county name
resource_id = "d5cd6a23-829c-426d-a63e-689a55a3db9c"

sql = f'''
SELECT * FROM "{resource_id}"
WHERE "County" = 'San Diego'
AND "DateofAdvisory"::date >= '2025-01-01'
'''

url = f"https://data.ca.gov/api/3/action/datastore_search_sql?sql={urllib.parse.quote(sql)}"
fileobj = urllib.request.urlopen(url)
response_dict = json.loads(fileobj.read())

beach_closures = pd.DataFrame(response_dict['result']['records'])
beach_closures_sedf = sample_data.spatial.from_xy(beach_closures, x_column='Station_UpperLon', y_column = 'Station_UpperLat')

gdb_path = aprx.defaultGeodatabase
beach_closures_sedf.spatial.to_featureclass(location=fr"{gdb_path}\beach_closures")

map.addDataFromPath(fr"{gdb_path}\beach_closures")

## Data Manipulation

In [139]:
# Get a list of beaches that have less than 20 closures, we don't care about those
counts = beach_closures.groupby('Beach_Name').size()
lowcount = counts[counts < 20].index.tolist()
# print(lowcount)

lowcount_str = ", ".join(f"'{name}'" for name in lowcount)
where_clause = f"Beach_Name NOT IN ({lowcount_str})"

In [138]:
# Copy the list and enter it manually to address apostrophes
where_clause = (
    "Beach_Name NOT IN ('All_SanDiego_County', 'Border Field State Park', 'Buccaneer Beach',"
    "'Cardiff State Beach', 'Carlsbad State Beach', 'Carlsbad municipal beach',"
    "'Coronado Cays (NR)', 'Dog Beach, O.B.', 'Harbor Beach', 'La Jolla Cove',"
    "'La Jolla Shores Beach', 'Marine Street Beach', 'Mission Bay',"
    "'Mission Bay, Campland On The Bay', 'Mission Bay, De Anza Cove',"
    "'Mission Bay, Fanuel Park', 'Mission Bay, Leisure Lagoon', 'Mission Bay, Mariners Basin',"
    "'Mission Bay, Sail Bay', 'Mission Bay, Vacation Isle',"
    "'Mission Bay, Ventura Cove', 'Mission Beach', 'Moonlight Beach',"
    "'Ocean Beach', 'Oceanside municipal beach, other', 'Pacific Beach',"
    "'Powerhouse Park 15th Street', 'San Dieguito River Beach',"
    "'San Onofre State Beach', 'South Carlsbad State Beach', 'Swami''s Park',"
    "'Tecolote Shores', 'Torrey Pines State Beach', 'Tourmaline Surfing Park',"
    "'USMC Camp Pendleton', 'WindanSea Beach', 'north Imperial Beach',"
    "'north county beaches')"
)
map.listLayers()[1].definitionQuery = where_clause